RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from pathlib import Path

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 2 PDF files to process

Processing: mid.pdf
  ✓ Loaded 3 pages

Processing: _Offer_Letter_B_Pavankumar.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [4]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 4 documents into 2 chunks

Example chunk:
Content: Dear B Pavankumar,
Congratulations! Your commitment to innovation and your collaborative spirit have made you
an ideal fit for our team. We are pleased to welcome you as the Web Development Team
Membe...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-09-26T15:35:56+00:00', 'title': 'Offer_Letter_B_Pavankumar', 'moddate': '2026-09-26T15:35:56+00:00', 'keywords': 'DAHWTRZ2fIw,BAG2Z-sGynI', 'author': 'Ramyaa D', 'trapped': '/False', 'source': '..\\data\\pdf\\_Offer_Letter_B_Pavankumar.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '_Offer_Letter_B_Pavankumar.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-09-26T15:35:56+00:00', 'title': 'Offer_Letter_B_Pavankumar', 'moddate': '2026-09-26T15:35:56+00:00', 'keywords': 'DAHWTRZ2fIw,BAG2Z-sGynI', 'author': 'Ramyaa D', 'trapped': '/False', 'source': '..\\data\\pdf\\_Offer_Letter_B_Pavankumar.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '_Offer_Letter_B_Pavankumar.pdf', 'file_type': 'pdf'}, page_content='Dear B Pavankumar,\nCongratulations! Your commitment to innovation and your collaborative spirit have made you\nan ideal fit for our team. We are pleased to welcome you as the Web Development Team\nMember of the Microsoft Learn Student Community (MLSC) at Vignan’s Institute of\nInformation Technology (VIIT). Your enthusiasm and dedication will be a valuable addition to\nour community and its future initiatives.\nAs a Core Team member, your leadership will be instrumental in building a vibrant and\nsupportive community. We are confident t